In [ ]:
%load_ext autoreload
%autoreload 2

import os
import tempfile
import numpy as np
import pandas as pd
import scanpy as sc
import rpy2.robjects as ro

from dotenv import load_dotenv; load_dotenv()

from utils import preprocessing
from utils import DEG
from utils.nebula_with_factors import filter_genes_for_nebula

%matplotlib inline

PARALLEL_NEBULA_SCRIPT_PATH = os.getenv("PARALLEL_NEBULA_SCRIPT")

# Config

In [ ]:
# ── Input ──────────────────────────────────────────────────────────────────
ADATA_PATH       = "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/SN/SN_combined_QC_mmc_ct-cured.h5ad"
CELL_TYPE        = "Microglia"          # None → use all cells
CT_VARIABLE      = "ct_for_deg"         # obs column with cell type labels

# ── Contrast ───────────────────────────────────────────────────────────────
CONTRAST_VARIABLE  = "condition"
CONTRAST_BASELINE  = "Control"
CONTRAST_STIM      = "XDP"
SAMPLE_VARIABLE    = "donor_id"
LIBRARY_SIZE_COL   = "total_counts"

# ── Formula covariates (excluding contrast — added automatically) ───────────
COVARIATES = ["age_of_death", "sex", "pct_counts_mt", "frac_intronic", "cohort"]

# ── Output ─────────────────────────────────────────────────────────────────
SAVE_FOLDER = f"/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/DEG/nebula_generic/{CELL_TYPE}"
os.makedirs(SAVE_FOLDER, exist_ok=True)

# Load adata

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)

# Use gene symbols as index
if "gene_symbol" in adata.var.columns:
    adata.var.index = adata.var["gene_symbol"]

# Drop bad cell types
if CT_VARIABLE in adata.obs.columns:
    adata = adata[~adata.obs[CT_VARIABLE].str.startswith("-")].copy()
    adata.obs[CT_VARIABLE] = adata.obs[CT_VARIABLE].str.replace(" ", "_")

# Drop samples with missing condition
adata = adata[adata.obs[CONTRAST_VARIABLE].notna()].copy()

# Subset to cell type if requested
if CELL_TYPE is not None:
    adata = adata[adata.obs[CT_VARIABLE] == CELL_TYPE].copy()

adata.X = adata.layers["counts"].copy()
print(adata)

# Filter genes

In [ ]:
adata = filter_genes_for_nebula(adata, SAMPLE_VARIABLE, CONTRAST_VARIABLE,
                                min_cells_per_sample=10, min_counts=50)

# Sanity check: enough samples per condition
MIN_SAMPLES = 4
counts = (
    adata.obs[[SAMPLE_VARIABLE, CONTRAST_VARIABLE]]
    .drop_duplicates()
    .groupby(CONTRAST_VARIABLE)[SAMPLE_VARIABLE]
    .count()
    .reindex([CONTRAST_BASELINE, CONTRAST_STIM], fill_value=0)
)
print(counts)
assert (counts >= MIN_SAMPLES).all(), f"Too few samples per condition (need >= {MIN_SAMPLES})"

# Drop cells with missing covariates

In [ ]:
all_covs = [CONTRAST_VARIABLE, *COVARIATES]
mask = adata.obs[all_covs].isna().any(axis=1)
print(f"Dropping {mask.sum()} cells with NaN covariates")
adata = adata[~mask].copy()

# Convert to .qs and run Nebula

In [ ]:
QS_PATH = f"{SAVE_FOLDER}/adata_for_nebula.qs"

tmp_dir = tempfile.mkdtemp(dir="/tmp")
adata.X = adata.layers["counts"].copy()
preprocessing.save_files_for_R_conversion(adata, tmp_dir)
preprocessing.build_qs_from_python_files(
    tmp_dir, saving_path=QS_PATH,
    contrast_var=CONTRAST_VARIABLE, reference_level=CONTRAST_BASELINE
)

In [ ]:
nebula_result_path = DEG.run_nebula_parallel_script(
    path_qs             = QS_PATH,
    path_nebula_script  = PARALLEL_NEBULA_SCRIPT_PATH,
    id_col              = SAMPLE_VARIABLE,
    covs                = [CONTRAST_VARIABLE, *COVARIATES],
    offset_col          = LIBRARY_SIZE_COL,
    n_folds             = 40,
    n_cores             = 40,
    save_tmp            = False,
)

# Save results

In [ ]:
save_csv_path = f"{SAVE_FOLDER}/nebula_results.csv"
ro.r(f'''
    library(qs)
    obj <- qread("{nebula_result_path}")
    df  <- obj$summary
    write.csv(df, "{save_csv_path}", row.names=FALSE)
''')

df_results = pd.read_csv(save_csv_path)
print(f"Saved {len(df_results)} genes to {save_csv_path}")
df_results.head()

In [ ]:
# Clean up tmp .qs (optional)
os.remove(QS_PATH)